# TP1 — Machine Learning: Churn NeoStream

Notebook organizado por exercício e tarefa. Os scripts `.py` pedidos são gerados pelas próprias células (`%%writefile`) e executados aqui mesmo, com as saídas visíveis.

## Preparação dos dados

In [13]:
import numpy as np
import pandas as pd


def gerar_base(caminho="assinantes_neostream.csv"):
    rng = np.random.default_rng(7)
    n = 12000

    plano = rng.choice(["basico", "padrao", "premium"], size=n, p=[0.4, 0.4, 0.2])
    metodo_pagamento = rng.choice(["cartao_credito", "boleto", "pix"], size=n, p=[0.55, 0.20, 0.25])
    dispositivo_principal = rng.choice(["smart_tv", "celular", "computador", "tablet"], size=n, p=[0.35, 0.35, 0.20, 0.10])
    tempo_assinatura_meses = np.round(np.clip(rng.exponential(scale=14, size=n), 0, 72), 1)
    uso_medio_semanal_horas = np.round(np.clip(rng.normal(9, 4, n), 0, 40), 1)
    numero_reclamacoes = rng.poisson(0.4, n)

    # churn: menor tempo de assinatura, menor uso e mais reclamações aumentam a chance de cancelar
    logit_churn = (
        -0.85
        - 0.04 * tempo_assinatura_meses
        - 0.06 * uso_medio_semanal_horas
        + 0.5 * numero_reclamacoes
        + rng.normal(0, 0.6, n)
    )
    prob_churn = 1 / (1 + np.exp(-logit_churn))
    churn = (rng.uniform(0, 1, n) < prob_churn).astype(int)

    # ltv_alto: região (não um corte linear) no espaço tempo x uso, com 5% de rótulos invertidos
    raio = np.sqrt(((tempo_assinatura_meses - 40) / 20) ** 2 + ((uso_medio_semanal_horas - 14) / 8) ** 2)
    ltv_alto = (raio < 1.0).astype(int)
    inverter = rng.random(n) < 0.05
    ltv_alto = np.where(inverter, 1 - ltv_alto, ltv_alto)

    df = pd.DataFrame({
        "cliente_id": [f"AST-{i:05d}" for i in range(1, n + 1)],
        "plano": plano,
        "metodo_pagamento": metodo_pagamento,
        "dispositivo_principal": dispositivo_principal,
        "tempo_assinatura_meses": tempo_assinatura_meses,
        "uso_medio_semanal_horas": uso_medio_semanal_horas,
        "numero_reclamacoes": numero_reclamacoes,
        "churn": churn,
        "ltv_alto": ltv_alto,
    })
    df.to_csv(caminho, index=False)
    return df


base = gerar_base()
print(base.shape)
print(f"Taxa de churn: {base['churn'].mean():.2%}")
base.head()

(12000, 9)
Taxa de churn: 17.56%


,cliente_id,plano,metodo_pagamento,dispositivo_principal,tempo_assinatura_meses,uso_medio_semanal_horas,numero_reclamacoes,churn,ltv_alto
0,AST-00001,padrao,pix,smart_tv,14.3,10.1,0,0,0
1,AST-00002,premium,cartao_credito,smart_tv,0.8,5.6,0,0,0
2,AST-00003,padrao,cartao_credito,computador,7.3,6.8,0,0,0
3,AST-00004,basico,pix,computador,10.3,4.6,0,1,0
4,AST-00005,basico,boleto,smart_tv,2.6,5.8,0,0,0


## Exercício 1 — Tipo de aprendizado e modo de treinamento

### Tarefa 1 — Tipo de aprendizado, Tarefa 2 — Modo de treinamento, Tarefa 3 — Tabela

| problema | tipo_de_aprendizado | modo_de_treinamento | justificativa |
|---|---|---|---|
| Prever quem vai cancelar no próximo mês | Supervisionado | Batch | Há rótulo histórico (cancelou / não cancelou), então é classificação supervisionada. O alvo é mensal e os dados chegam em lotes, sem exigência de resposta imediata: retreina-se periodicamente com o histórico completo. |
| Agrupar assinantes por padrão de consumo | Não supervisionado | Batch | Não existe rótulo prévio de grupo; o algoritmo (ex.: K-Means) descobre a estrutura só com as features de consumo. A segmentação é analítica e atualizada em ciclos (semanal/mensal), sem necessidade de tempo real. |
| Recomendar séries em tempo real conforme o assinante assiste | Não supervisionado | Online | Não há uma "série correta" rotulada; as recomendações vêm de similaridade e co-consumo entre assinantes e títulos (filtragem colaborativa). Cada nova visualização chega continuamente e a resposta precisa ser imediata, então o modelo se atualiza de forma incremental. |
| Ajustar o preço do plano promocional a cada reação do mercado | Não supervisionado* | Online | Não há rótulo de "preço certo" a priori: o sinal é a reação do mercado (receita, adesão) depois de cada decisão. Os dados chegam a cada interação e o preço deve ser atualizado a cada nova reação, o que exige aprendizado online. |

\* Estritamente, este caso é **aprendizado por reforço** (agente que age e recebe recompensa). Dentro da dicotomia supervisionado × não supervisionado pedida no exercício, ele fica do lado sem rótulos.

## Exercício 2 — Bias e variância dos protótipos

### Tarefa 1 — Diagnóstico de cada protótipo

Como a base foi amostrada com o mesmo número de exemplos por classe, o chute aleatório rende cerca de 50%, então acurácia é uma métrica confiável aqui.

| protótipo | treino | teste | gap | diagnóstico |
|---|---|---|---|---|
| 1 | 98% | 61% | 37 p.p. | **Variância alta** (overfitting), bias baixo |
| 2 | 71% | 69% | 2 p.p. | **Bias alto** (underfitting), variância baixa |

- **Protótipo 1:** quase não erra no treino (bias baixo), mas perde 37 pontos no teste. O modelo memorizou ruído e particularidades do treino, e o desempenho é muito sensível à amostra (variância alta).
- **Protótipo 2:** treino e teste praticamente iguais, então generaliza de forma estável (variância baixa). Porém 71% no treino mostra que o modelo não captura bem o padrão nem nos dados que ele já viu: capacidade insuficiente (bias alto).

### Tarefa 2 — Regularização e o tradeoff bias-variância

Regularizar penaliza a complexidade do modelo: **reduz variância e aumenta bias**.

- No protótipo 1, o ganho em variância supera o custo em bias. Como o bias já é mínimo (98% no treino), aceitar um pouco mais de bias derruba o gap e tende a **subir** o acerto de teste.
- No protótipo 2, a variância já é baixa e quase não há o que reduzir. Mais regularização só aumentaria um bias que já é o problema, empurrando treino e teste para baixo, o que **prejudica** o modelo.

### Tarefa 3 — Recomendação

- **Protótipo 1: mais regularização** (L1/L2 mais forte, menor profundidade/menos parâmetros, early stopping), pois o erro vem de variância.
- **Protótipo 2: menos regularização** (e/ou mais capacidade: mais features, interações, modelo mais flexível), pois o erro vem de bias.

Justificativa: o gap treino-teste mostra que o erro do 1 é de variância e o do 2 é de bias. A regularização só corrige variância, então ela ajuda um e atrapalha o outro. Nota: o 2 é o mais estável, mas 69% de teste é menor que o potencial do problema; o caminho é ganhar capacidade sem reabrir o gap do protótipo 1.

## Exercício 3 — Divisão estratificada

Funções de carga e divisão ficam em `preparo_dados.py` para serem reutilizadas nos exercícios seguintes. Apenas as colunas listadas em `COLUNAS_NUMERICAS` e `COLUNAS_CATEGORICAS` entram como features: `cliente_id` (identificador) e `ltv_alto` (alvo do TP2) são descartados pelo `ColumnTransformer`.

### Tarefa 1 e 2 — Carga, alvo e `train_test_split` estratificado

In [14]:
%%writefile preparo_dados.py
import pandas as pd
from sklearn.model_selection import train_test_split

ARQUIVO = "assinantes_neostream.csv"
ALVO = "churn"
COLUNAS_NUMERICAS = ["tempo_assinatura_meses", "uso_medio_semanal_horas", "numero_reclamacoes"]
COLUNAS_CATEGORICAS = ["plano", "metodo_pagamento", "dispositivo_principal"]


def carregar_dados(caminho=ARQUIVO):
    return pd.read_csv(caminho)


def separar_alvo(df):
    return df.drop(columns=[ALVO]), df[ALVO]


def dividir_treino_teste(X, y):
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

Overwriting preparo_dados.py


In [15]:
from preparo_dados import carregar_dados, separar_alvo, dividir_treino_teste

df = carregar_dados()
X, y = separar_alvo(df)
X_train, X_test, y_train, y_test = dividir_treino_teste(X, y)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (9600, 8) | X_test: (2400, 8)


### Tarefa 3 — Proporção de churn nos dois conjuntos

In [16]:
def comparar_proporcoes(y_treino, y_teste):
    return pd.DataFrame({
        "treino": y_treino.value_counts(normalize=True),
        "teste": y_teste.value_counts(normalize=True),
    }).rename_axis("churn")


comparar_proporcoes(y_train, y_test).round(4)

,treino,teste
churn,,
0,0.8244,0.8246
1,0.1756,0.1754


Comparação registrada: a proporção de churn é praticamente idêntica nos dois conjuntos (≈ 17,6% de cancelamentos em treino e em teste, a mesma da base completa), efeito direto do `stratify=y`.

### Tarefa 4 — Por que o fit só depois da divisão

Scaler e encoder aprendem estatísticas (média, desvio, categorias) dos dados em que são ajustados; se o fit usar a base inteira, informação do conjunto de teste entra no treino e a avaliação deixa de ser honesta.

## Exercício 4 — `ColumnTransformer`

### Tarefas 1, 2 e 3 — Construção, fit apenas no treino e `preprocessamento_heterogeneo.py`

In [17]:
%%writefile preprocessamento_heterogeneo.py
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from preparo_dados import (
    COLUNAS_CATEGORICAS,
    COLUNAS_NUMERICAS,
    carregar_dados,
    dividir_treino_teste,
    separar_alvo,
)


def construir_preprocessador():
    return ColumnTransformer([
        ("numericas", StandardScaler(), COLUNAS_NUMERICAS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORICAS),
    ])


def main():
    X, y = separar_alvo(carregar_dados())
    X_train, X_test, _, _ = dividir_treino_teste(X, y)

    preprocessador = construir_preprocessador()
    X_train_transformado = preprocessador.fit_transform(X_train)
    X_test_transformado = preprocessador.transform(X_test)

    print("Shape de X_train transformado:", X_train_transformado.shape)
    print("Shape de X_test transformado:", X_test_transformado.shape)


if __name__ == "__main__":
    main()

Overwriting preprocessamento_heterogeneo.py


In [18]:
%run preprocessamento_heterogeneo.py

Shape de X_train transformado: (9600, 13)
Shape de X_test transformado: (2400, 13)


As 13 colunas são 3 numéricas padronizadas + 3 (plano) + 3 (método de pagamento) + 4 (dispositivo) do one-hot. O `fit_transform` roda só em `X_train`; `X_test` recebe apenas `transform`.

## Exercício 5 — `Pipeline` com `LogisticRegression`

### Tarefas 1, 2 e 3 — Pipeline, `fit` único, previsões e `pipeline_classificacao.py`

In [19]:
%%writefile pipeline_classificacao.py
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

from preparo_dados import carregar_dados, dividir_treino_teste, separar_alvo
from preprocessamento_heterogeneo import construir_preprocessador


def construir_pipeline():
    return Pipeline([
        ("preprocessador", construir_preprocessador()),
        ("classificador", LogisticRegression(max_iter=1000)),
    ])


def main():
    X, y = separar_alvo(carregar_dados())
    X_train, X_test, y_train, y_test = dividir_treino_teste(X, y)

    pipeline = construir_pipeline()
    pipeline.fit(X_train, y_train)
    previsoes = pipeline.predict(X_test)

    print("Previsões geradas:", previsoes.shape[0])
    print(f"Proporção prevista de churn: {previsoes.mean():.2%}")
    print(classification_report(y_test, previsoes, digits=3))


if __name__ == "__main__":
    main()

Overwriting pipeline_classificacao.py


In [20]:
%run pipeline_classificacao.py

Previsões geradas: 2400
Proporção prevista de churn: 0.38%
              precision    recall  f1-score   support

           0      0.826     0.998     0.904      1979
           1      0.556     0.012     0.023       421

    accuracy                          0.825      2400
   macro avg      0.691     0.505     0.464      2400
weighted avg      0.779     0.825     0.749      2400



Observação: com o limiar padrão de 0,5 o modelo quase não prevê churn (recall da classe 1 ≈ 1%), efeito da classe minoritária (~18%) e de features com sinal fraco (AUC ≈ 0,66). Acurácia de 82,5% é praticamente a taxa de quem não cancela, por isso o AUC é a métrica usada daqui em diante.

### Tarefa 4 — Pipeline e data leakage

O Pipeline faz o `fit` do pré-processamento somente nos dados que recebe em `fit` e aplica apenas `transform` com esses mesmos parâmetros em `predict`, então nada do conjunto de teste (ou do lote novo) influencia o ajuste. Além disso, treino e produção passam exatamente pelo mesmo objeto, eliminando divergência entre as transformações aplicadas em cada ambiente.

## Exercício 6 — `GridSearchCV` sobre o Pipeline

Como só ~18% dos assinantes cancelam, acurácia é enganosa (prever "ninguém cancela" já dá ~82%). Por isso a busca usa `roc_auc` como métrica de validação cruzada.

### Tarefas 1, 2 e 3 — Grade de `C`, busca com `cv=5` e melhores resultados

In [21]:
from sklearn.model_selection import GridSearchCV

from pipeline_classificacao import construir_pipeline

GRADE_C = {"classificador__C": [0.01, 0.1, 1, 10, 100]}


def buscar_melhor_c(pipeline, X_treino, y_treino):
    busca = GridSearchCV(pipeline, GRADE_C, cv=5, scoring="roc_auc", n_jobs=-1)
    return busca.fit(X_treino, y_treino)


busca = buscar_melhor_c(construir_pipeline(), X_train, y_train)

print("best_params_:", busca.best_params_)
print(f"best_score_ (ROC AUC, cv=5): {busca.best_score_:.4f}")

(pd.DataFrame(busca.cv_results_)[["param_classificador__C", "mean_test_score", "std_test_score"]]
 .rename(columns={"param_classificador__C": "C", "mean_test_score": "auc_medio", "std_test_score": "desvio"})
 .round(4))

best_params_: {'classificador__C': 0.01}
best_score_ (ROC AUC, cv=5): 0.6578


,C,auc_medio,desvio
0,0.01,0.6578,0.0095
1,0.10,0.6575,0.0097
2,1.00,0.6574,0.0097
3,10.00,0.6574,0.0097
4,100.00,0.6574,0.0097


Leitura do resultado: `best_params_` aponta `C = 0.01`, mas os cinco valores de `C` ficam praticamente empatados (AUC médio ≈ 0,657, diferenças menores que o desvio entre folds de ≈ 0,01). Ou seja, a regularização quase não muda a métrica nesta base; o ganho de C=0.01 é marginal e a escolha é a mais simples (mais regularizada) entre modelos equivalentes.

## Exercício 7 — Auditoria de data leakage

### Tarefa 1 — Onde ocorre o vazamento

O vazamento está no `StandardScaler.fit` aplicado ao `X` completo **antes** do `train_test_split`. Nesse ponto a média e o desvio padrão já incorporam as linhas que viram teste, então o treino é escalado com estatísticas que "conhecem" o teste. O modelo é avaliado em dados que influenciaram o pré-processamento, o que torna a métrica reportada otimista: ela não simula um dado realmente novo, como o que chega em produção.

### Tarefa 2 — Fluxo correto

**Fluxo incorreto (do estagiário)**
```
X_escalado = StandardScaler().fit_transform(X)        # fit com treino + teste
X_train, X_test, ... = train_test_split(X_escalado, y)
modelo.fit(X_train, y_train); avaliar em X_test
```

**Fluxo corrigido**
```
X_train, X_test, ... = train_test_split(X, y, stratify=y)   # primeiro a divisão
scaler.fit(X_train)                                         # fit só no treino
modelo.fit(scaler.transform(X_train), y_train)
avaliar em scaler.transform(X_test)                         # só transform no teste
```

### Tarefa 3 — Documentação da auditoria com evidência

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from preparo_dados import COLUNAS_NUMERICAS


def auc_do_modelo(X_tr, X_te, y_tr, y_te):
    modelo = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, modelo.predict_proba(X_te)[:, 1])


def fluxo_incorreto(X_num, y):
    scaler = StandardScaler().fit(X_num)
    X_tr, X_te, y_tr, y_te = train_test_split(scaler.transform(X_num), y, test_size=0.2, stratify=y, random_state=42)
    return scaler, auc_do_modelo(X_tr, X_te, y_tr, y_te)


def fluxo_correto(X_num, y):
    X_tr, X_te, y_tr, y_te = dividir_treino_teste(X_num, y)
    scaler = StandardScaler().fit(X_tr)
    return scaler, auc_do_modelo(scaler.transform(X_tr), scaler.transform(X_te), y_tr, y_te)


X_num = X[COLUNAS_NUMERICAS]
scaler_vazado, auc_incorreto = fluxo_incorreto(X_num, y)
scaler_limpo, auc_correto = fluxo_correto(X_num, y)

estatisticas = pd.DataFrame({
    "media_vazada": scaler_vazado.mean_, "media_limpa": scaler_limpo.mean_,
    "desvio_vazado": scaler_vazado.scale_, "desvio_limpo": scaler_limpo.scale_,
}, index=COLUNAS_NUMERICAS)

display(estatisticas.round(4))
print(f"AUC no teste, fluxo incorreto: {auc_incorreto:.4f}")
print(f"AUC no teste, fluxo correto:   {auc_correto:.4f}")

,media_vazada,media_limpa,desvio_vazado,desvio_limpo
tempo_assinatura_meses,13.8582,13.8698,13.5660,13.5587
uso_medio_semanal_horas,9.0383,9.0239,3.9640,3.9716
numero_reclamacoes,0.3948,0.3960,0.6266,0.6255


AUC no teste, fluxo incorreto: 0.6637
AUC no teste, fluxo correto:   0.6637


**Leitura da evidência.** A tabela mostra que as estatísticas do scaler mudam quando o teste participa do fit: esse é o vazamento, mensurável. Já o AUC é idêntico nos dois fluxos (0,6637) neste dataset, porque a padronização usa só duas estatísticas globais e a base tem 12 mil linhas. Isso não torna o fluxo aceitável: o erro é de método e o impacto cresce com transformações que aprendem mais com os dados (seleção de features, PCA, imputação, target encoding) ou com bases pequenas. Corrigir a ordem é o que garante que a métrica represente o desempenho em produção.

## Exercício 8 — Script final `pipeline_final_churn.py`

### Tarefas 1, 2 e 3 — Integração dos Exercícios 3 a 6, ajuste final e decisões de design

In [23]:
%%writefile pipeline_final_churn.py
"""
Pipeline final de classificação de churn (NeoStream).

Decisões de design
------------------
1. Split estratificado (stratify=y): o churn é ~18% da base; sem estratificar,
   treino e teste poderiam ter proporções diferentes e a métrica ficaria instável.
2. Split ANTES de qualquer fit: scaler/encoder só podem aprender com o treino,
   senão o teste vaza informação para o modelo (data leakage).
3. ColumnTransformer: as features são heterogêneas (numéricas e categóricas) e
   cada tipo exige um transformador; um único objeto organiza tudo e descarta
   colunas que não são features (cliente_id, ltv_alto).
4. OneHotEncoder(handle_unknown="ignore"): categorias inéditas em produção não
   quebram a previsão.
5. Pipeline em vez de passos manuais: pré-processamento e modelo viram um único
   objeto, treinado e aplicado do mesmo jeito em treino, validação e produção.
6. GridSearchCV sobre o Pipeline (parâmetro classificador__C): o pré-processamento
   é re-ajustado dentro de cada fold, então a validação cruzada também não vaza.
7. Métrica roc_auc: com ~18% de churn, acurácia seria enganosa.
8. Pipeline final salvo com joblib: o artefato único é o que vai para o deploy.
"""
import joblib
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ARQUIVO = "assinantes_neostream.csv"
ARQUIVO_MODELO = "pipeline_churn.joblib"
ALVO = "churn"
COLUNAS_NUMERICAS = ["tempo_assinatura_meses", "uso_medio_semanal_horas", "numero_reclamacoes"]
COLUNAS_CATEGORICAS = ["plano", "metodo_pagamento", "dispositivo_principal"]
GRADE_C = {"classificador__C": [0.01, 0.1, 1, 10, 100]}


def dividir_dados(caminho=ARQUIVO):
    df = pd.read_csv(caminho)
    X, y = df.drop(columns=[ALVO]), df[ALVO]
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


def construir_pipeline():
    preprocessador = ColumnTransformer([
        ("numericas", StandardScaler(), COLUNAS_NUMERICAS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORICAS),
    ])
    return Pipeline([
        ("preprocessador", preprocessador),
        ("classificador", LogisticRegression(max_iter=1000)),
    ])


def buscar_hiperparametros(pipeline, X_train, y_train):
    busca = GridSearchCV(pipeline, GRADE_C, cv=5, scoring="roc_auc", n_jobs=-1)
    return busca.fit(X_train, y_train)


def treinar_pipeline_final(pipeline, melhores_parametros, X_train, y_train):
    return clone(pipeline).set_params(**melhores_parametros).fit(X_train, y_train)


def avaliar(pipeline, X_test, y_test):
    previsoes = pipeline.predict(X_test)
    print(f"ROC AUC no teste: {roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1]):.4f}")
    print(classification_report(y_test, previsoes, digits=3))


def main():
    X_train, X_test, y_train, y_test = dividir_dados()

    pipeline = construir_pipeline()
    busca = buscar_hiperparametros(pipeline, X_train, y_train)
    print("Melhores parâmetros:", busca.best_params_)
    print(f"ROC AUC (cv=5): {busca.best_score_:.4f}")

    pipeline_final = treinar_pipeline_final(pipeline, busca.best_params_, X_train, y_train)
    avaliar(pipeline_final, X_test, y_test)

    joblib.dump(pipeline_final, ARQUIVO_MODELO)
    print(f"Pipeline salvo em {ARQUIVO_MODELO}")


if __name__ == "__main__":
    main()

Overwriting pipeline_final_churn.py


In [24]:
%run pipeline_final_churn.py

Melhores parâmetros: {'classificador__C': 0.01}
ROC AUC (cv=5): 0.6578
ROC AUC no teste: 0.6640
              precision    recall  f1-score   support

           0      0.825     0.999     0.904      1979
           1      0.667     0.005     0.009       421

    accuracy                          0.825      2400
   macro avg      0.746     0.502     0.457      2400
weighted avg      0.797     0.825     0.747      2400

Pipeline salvo em pipeline_churn.joblib


### Tarefa 4 — Por que essa organização é mais segura para produção

Ao concentrar divisão estratificada, `ColumnTransformer`, modelo e busca de hiperparâmetros em um único `Pipeline` treinado com `fit`, o pré-processamento é ajustado apenas com o treino (e apenas com os folds de treino dentro do `GridSearchCV`), o que impede vazamento de dados e mantém a métrica honesta. Em produção, o mesmo artefato serializado recebe o dado bruto e aplica exatamente as transformações aprendidas, sem etapas manuais que possam ser esquecidas, reordenadas ou reimplementadas de forma diferente, e o `handle_unknown="ignore"` evita falhas com categorias novas. O resultado é um objeto único, reproduzível e fácil de revisar e versionar na esteira de deploy.